<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/RAG_Completo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pypdf
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.2/382.2 kB 6.8 MB/s eta 0:00:00


In [2]:
!pip install -q openai

In [3]:
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

print("API Key cargada:", api_key is not None)

API Key cargada: True


In [4]:
from openai import OpenAI
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(
    api_key=api_key
)

In [5]:
from google.colab import files

uploaded = files.upload()

pdf_path = list(
    uploaded.keys()
)[0]

print(pdf_path)

Saving documento_prueba_ia_pdf_machine_learning.pdf to documento_prueba_ia_pdf_machine_learning.pdf
documento_prueba_ia_pdf_machine_learning.pdf


In [6]:
#Extrae el texto
from pypdf import PdfReader

reader = PdfReader(
    pdf_path
)

paginas = []

for numero, pagina in enumerate(
    reader.pages,
    start=1
):

    texto = pagina.extract_text()

    if texto:

        paginas.append({
            "pagina": numero,
            "texto": texto
        })

print(
    "Páginas:",
    len(paginas)
)

Páginas: 3


In [7]:
#Divide el Texto
chunks = []

for pagina in paginas:

    texto = pagina["texto"]

    tamaño = 800

    for i in range(
        0,
        len(texto),
        tamaño
    ):

        fragmento = texto[
            i:i+tamaño
        ]

        chunks.append({
            "pagina": pagina["pagina"],
            "texto": fragmento
        })

print(
    "Fragmentos:",
    len(chunks)
)

Fragmentos: 6


In [8]:
#Embeddings
from sentence_transformers import SentenceTransformer

model_embedding = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

textos = [
    c["texto"]
    for c in chunks
]

embeddings = model_embedding.encode(
    textos,
    show_progress_bar=True
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.8 MB/s eta 0:00:00


In [10]:
#Faiss

import faiss
import numpy as np

embeddings = np.array(
    embeddings,
    dtype="float32"
)

index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(
    embeddings
)

In [11]:
def buscar_pdf(
    pregunta,
    k=1
):

    vector = model_embedding.encode(
        [pregunta]
    )
    vector = np.array(
        vector,
        dtype="float32"
    )

    distancias, indices = index.search(
        vector,
        k
    )

    resultados = []

    for distancia, indice in zip(
        distancias[0],
        indices[0]
    ):
        resultados.append({
            "pagina":
                chunks[indice]["pagina"],

            "texto":
                chunks[indice]["texto"],

            "distancia":
                float(distancia)
        })

    return resultados

In [12]:
def obtener_contexto(pregunta, k=5):

    resultados = buscar_pdf(
        pregunta,
        k=k
    )

    contexto = ""

    for i, r in enumerate(resultados):

        contexto += f"""
--- Fragmento {i+1} ---
Página: {r['pagina']}

{r['texto']}
"""

    return contexto, resultados

In [21]:
def preguntar_rag(
    pregunta,
    k=5
):

    contexto, resultados = obtener_contexto(
        pregunta,
        k=k
    )

    prompt = f"""
Responde la pregunta utilizando únicamente
la información del contexto.

Pregunta:
{pregunta}

Contexto:
{contexto}

Reglas:

1. No inventes información.
2. Utiliza únicamente el contexto.
3. Si no existe información suficiente,
   dilo claramente.
4. Responde en español.
5. Explica la respuesta de forma clara.
"""

    respuesta = client.responses.create(

        model="gpt-5.4-mini",

        instructions="""
Eres un asistente de análisis documental.
Tu trabajo es responder preguntas utilizando
únicamente la información recuperada de un PDF.
""",

        input=prompt
    )

    paginas = sorted(
        set(
            r["pagina"]
            for r in resultados
        )
    )

    return respuesta.output_text, paginas

In [26]:
pregunta = input(
    "Pregunta al PDF: "
)

respuesta, paginas = preguntar_rag(
    pregunta
)

print("\nRESPUESTA:\n")

print(respuesta)

print("\nInformación recuperada de las páginas:")

print(
    ", ".join(
        str(p)
        for p in paginas
    )
)

Pregunta al PDF: Dame las técnicas de machine learning

RESPUESTA:

Las técnicas o algoritmos de machine learning mencionados en el contexto son:

- Regresión lineal
- Árbol de decisión
- Random Forest
- K-Means
- Red neuronal

Si quieres, también puedo clasificarlos según si se usan para regresión, clasificación o agrupamiento.

Información recuperada de las páginas:
1, 2, 3
